# Simultaneous Turns — Contested Resources & Retry Policy

A companion to **`06_simultaneous_turns.ipynb`**. That notebook introduced the
opt-in **simultaneous** turn mode (`Game(..., turn_mode="simultaneous")`): every NPC
decides against the same *turn-start snapshot*, then the gathered commands resolve one at
a time in priority order. This notebook shows what happens when those choices **collide**
— two characters reaching for the same thing — and how the engine settles it
(**issue #42**).

The motivating case from the issue: *two characters in a kitchen both decide to grab the
banana; one gets there first, so the other's grab can no longer succeed.* Rather than
letting the loser hit a confusing generic failure (`"I don't see it."`), the resolve phase
now:

1. **claims** — each gathered command records the resource it reaches for (the banana);
2. **arbitrates** — the higher-priority character wins it, decided by an explainable
   ordering *policy*, not by who happened to mutate the world first;
3. **recovers** — the loser is told *why* it lost and either takes a **ranked fallback**
   it pre-chose, or **reflects** on the real reason and decides again; and
4. **records** — the collision is logged as a first-class `conflict` event.

This mirrors the "game master adjudicates agent actions" idea from DeepMind's
[Concordia](https://arxiv.org/abs/2312.03664). It runs **fully offline** with tiny
`ScriptedAgent` rules — no API key. The full design is in
`docs/design/simultaneous-actions.md`.

> **Run this from the `notebooks/` directory**, with the repo installed
> (`pip install -e ".[dev]"`).

In [1]:
from rich.console import Console

from text_adventure_games import games, things
from text_adventure_games.npc import ScriptedAgent
from text_adventure_games.reporting import NORMAL, RichTerminalRenderer
from text_adventure_games.turns import DEFAULT_PHASES


def force_rich_notebook_output(game, level=NORMAL):
    """Use Rich's Jupyter renderer for parser output in this notebook."""
    game.parser.set_renderer(
        RichTerminalRenderer(level=level, console=Console(force_jupyter=True, width=100))
    )
    return game


# --- Three tiny scripted "brains". Each is just a rule (observation) -> command.
# The observation is the text the engine shows the NPC; items appear as "* banana".

def plain_grab(item):
    """Always reach for `item` when it's in view (our eager winner)."""
    def rule(observation):
        return f"take {item}" if f"* {item}" in observation else None
    return rule


def ranked_grab(primary, *backups):
    """Decide a RANKED LIST up front: the primary, then backups. If the loser of
    the primary pre-chose backups, it takes the first workable one immediately —
    no second decision (the cheap 'fallback intents' arm)."""
    def rule(observation):
        if f"* {primary}" in observation:
            return [f"take {primary}", *[f"take {b}" for b in backups]]
        return None
    return rule


def reflective_grab(primary, *backups):
    """Decide ONE command. If a conflict reflection later says we lost, pivot to
    a backup that's still available (the 'informed retry' arm — one extra
    decision, seeded with the true reason)."""
    def rule(observation):
        if "first this turn" in observation:        # the conflict reflection
            for alt in backups:
                if f"* {alt}" in observation:
                    return f"take {alt}"
            return None
        return f"take {primary}" if f"* {primary}" in observation else None
    return rule


def build_kitchen(scullion_agent):
    """A one-room scene: a banana (everyone's favorite) and an apple, plus two
    hungry NPCs. The cook is quicker on the draw (initiative 3 vs 1)."""
    kitchen = things.Location("Kitchen", "A warm castle kitchen smelling of bread.")
    kitchen.add_item(things.Item("banana", "a ripe banana"))
    kitchen.add_item(things.Item("apple", "a crisp apple"))

    player = things.Character("player", "You are a hungry traveler.", "I just watch.")
    cook = things.Character("cook", "the head cook", "That banana is mine.")
    scullion = things.Character("scullion", "a kitchen hand", "I want the banana too.")
    cook.set_property("initiative", 3)        # quicker
    scullion.set_property("initiative", 1)

    game = games.Game(
        kitchen, player, characters=[cook, scullion], turn_mode="simultaneous"
    )
    kitchen.add_character(cook)
    kitchen.add_character(scullion)
    force_rich_notebook_output(game)

    cook.set_agent(ScriptedAgent(plain_grab("banana")))
    scullion.set_agent(scullion_agent)
    return game


def show_inventories(game):
    for name in ("cook", "scullion"):
        held = list(game.characters[name].inventory) or ["nothing"]
        print(f"{name:9} holds: {', '.join(held)}")


def show_event_log(game):
    print("\nEvent log (the conflict's paper trail):")
    for e in game.events:
        print(f"  turn {e.turn:>2}  {e.actor:<9} {e.action:<10} {e.summary}")

## 1. A ranked fallback (the cheap arm)

Both NPCs decide `take banana` against the **same snapshot** — neither sees the other
move. But the **scullion** was prepared: its brain returns a *ranked list*
`["take banana", "take apple"]`. The head is what it wants; the tail is a backup it
pre-chose **at decide time**.

At resolve the cook (higher initiative) wins the banana. The scullion *loses the claim*,
so instead of blindly running its doomed `take banana`, it immediately falls back to the
apple — **no second decision needed**. The collision is recorded as a `conflict` event
naming the winner.

In [2]:
game = build_kitchen(ScriptedAgent(ranked_grab("banana", "apple")))
game.parser.echo_commands = True
game.do_command("look")   # the player just watches

print()
show_inventories(game)
show_event_log(game)

[player command] look

[narration] KITCHEN
            A warm castle kitchen smelling of bread.
            
            You see:
             * banana - a ripe banana
             * apple - a crisp apple
            Characters:
             * cook - the head cook
             * scullion - a kitchen hand
            

Turn 1 ─────────────────────────────────────────────────────────────────────────────────────────────

cook [action] take banana

[player command] take banana

...

[conflict] cook got the banana first this turn.

scullion [action] take apple

[player command] take apple

[narration] scullion got the apple.


cook      holds: banana
scullion  holds: apple

Event log (the conflict's paper trail):
  turn  0  player    describe   look
  turn  1  cook      get        take banana
  turn  1  scullion  conflict   cook got the banana first this turn.
  turn  1  scullion  get        take apple


See the `⚔` line: the scullion is told **`cook got the banana first this turn.`** — not
a phantom *"I don't see it."* It spends zero extra thinking and walks off with the apple,
and the event log carries a `conflict` row whose payload names the `winner`.

This is the cheapest recovery: the agent already ranked its options, so the loser's
fallback runs with **no extra LLM round-trip**.

## 2. An informed retry (when there's no fallback)

What if the loser *didn't* pre-rank a backup? Here the scullion returns a single command,
`take banana`. When it loses, the engine **reflects** on the *true* reason — "cook got the
banana first" — and lets the scullion **decide again** against the live kitchen. It sees
the banana is gone (but the apple isn't) and recovers.

Watch for the extra `reflect` line this time — that's the one additional decision the
fallback arm avoided.

In [3]:
game = build_kitchen(ScriptedAgent(reflective_grab("banana", "apple")))
game.parser.echo_commands = True
game.do_command("look")

print()
show_inventories(game)
show_event_log(game)

[player command] look

[narration] KITCHEN
            A warm castle kitchen smelling of bread.
            
            You see:
             * banana - a ripe banana
             * apple - a crisp apple
            Characters:
             * cook - the head cook
             * scullion - a kitchen hand
            

Turn 1 ─────────────────────────────────────────────────────────────────────────────────────────────

cook [action] take banana

[player command] take banana

...

scullion [reflection] cook got the banana first this turn.

scullion [action] take apple

[player command] take apple

[narration] scullion got the apple.


cook      holds: banana
scullion  holds: apple

Event log (the conflict's paper trail):
  turn  0  player    describe   look
  turn  1  cook      get        take banana
  turn  1  scullion  conflict   cook got the banana first this turn.
  turn  1  scullion  get        take apple


Same outcome (the scullion ends up with the apple), but a different *mechanism*: it
**re-decided** with the real reason in hand, instead of running a pre-chosen backup. Both
arms beat the old behavior, where the loser just saw a generic precondition failure and
learned nothing about *why*.

A note on honesty: the loser is only told it lost when a higher-priority character
**actually secured** the resource. If, say, the *player* had grabbed the banana first
(the player always resolves first), neither NPC would have "won" it — so they'd get a
plain failure, never a fabricated *"someone beat me to it."*

## 3. Ordering is a policy, not a fixed field

Who wins a contest is decided by the **resolve order**, and that order is more than a
single `initiative` field. With an opt-in **phase map** (`game.phases`), actions sort by
*kind* first — `communicate < move < manipulate < fight` — and only then by initiative.

Below, a slow **herald** (initiative 1) just announces, while the fast **cook**
(initiative 5) grabs. Because *speech* is an earlier phase than *manipulation*, the
herald's `say` resolves **first** — even though the cook is quicker. With no phase map the
faster cook would lead; phases let a game express "talk lands before anyone acts on it."
(This is also the groundwork for live agent-to-agent dialog — see the design doc §7.)

In [4]:
kitchen = things.Location("Kitchen", "A warm castle kitchen smelling of bread.")
kitchen.add_item(things.Item("banana", "a ripe banana"))
player = things.Character("player", "You are a hungry traveler.", "I just watch.")
herald = things.Character("herald", "a loud herald", "I announce things.")
cook = things.Character("cook", "the head cook", "That banana is mine.")
herald.set_property("initiative", 1)   # slower...
cook.set_property("initiative", 5)     # ...but the herald still speaks first

game = games.Game(kitchen, player, characters=[herald, cook], turn_mode="simultaneous")
kitchen.add_character(herald)
kitchen.add_character(cook)
force_rich_notebook_output(game)
game.parser.echo_commands = True

herald.set_agent(ScriptedAgent(lambda obs: "say I want that banana"))
cook.set_agent(ScriptedAgent(plain_grab("banana")))

game.phases = DEFAULT_PHASES   # opt in: communicate < move < manipulate < fight
game.do_command("look")

print("\nResolve order this round:")
for e in game.events:
    if e.turn == game.turn:
        print(f"  {e.actor:8} {e.action}")

[player command] look

[narration] KITCHEN
            A warm castle kitchen smelling of bread.
            
            You see:
             * banana - a ripe banana
            Characters:
             * herald - a loud herald
             * cook - the head cook
            

Turn 1 ─────────────────────────────────────────────────────────────────────────────────────────────

herald [action] say I want that banana

[player command] say I want that banana

[narration] herald says: I want that banana

cook [action] take banana

[player command] take banana

[narration] cook got the banana.


Resolve order this round:
  herald   say
  cook     get


## Where this fits

Issue #42 builds **stages 1–4** of `docs/design/simultaneous-actions.md`:

| | Stage | What it adds |
|---|---|---|
| ✅ | 1. Ordering seam | `resolve_order(intents, game)` with a tuple key (phase, initiative, gather order) |
| ✅ | 2. Phases | opt-in `Game.phases`; default `communicate < move < manipulate < fight` |
| ✅ | 3. Claim / arbitrate | detect contention, conflict-aware reasons, first-class `conflict` events |
| ✅ | 4. Fallback intents | `Agent.decide` may return a ranked `list[str]`; loser takes the next workable |

Deliberately **left for later**: deferred re-resolution of pending-prerequisite failures
(stage 5), live same-turn dialog (stage 6), and a fully order-independent transactional
commit (stage 7). Everything here is **opt-in and additive** — with no phase map, no
fallbacks, and no contention, a simultaneous round behaves exactly as it did before.